<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 💬 LangChain Message —— 定义 · 传参 · 流转速查

**一句话定位**:Message 是 chat model 的**唯一通用货币**——输入是 `list[Message]`,输出是 `AIMessage`,LangGraph 的 state 里存的也是它。

</div>

### 1️⃣ 五个核心类型(`langchain_core.messages`)

| 类 | `type` | 谁产生 | 关键字段 |
|---|---|---|---|
| `SystemMessage` | `system` | 开发者 | `content`(行为指令) |
| `HumanMessage` | `human` | 用户 | `content`(可多模态) |
| `AIMessage` | `ai` | 模型 | `tool_calls` · `usage_metadata` · `response_metadata` |
| `ToolMessage` | `tool` | 工具执行结果 | **`tool_call_id`**(必填) · `artifact` · `status` |
| `RemoveMessage` | `remove` | 开发者(改 state) | `id`(要删的消息 id) |

**公共字段**:`content` · `name`(区分同角色多参与者) · `id`(`add_messages` 靠它去重/覆盖/删除) · `additional_kwargs` · `response_metadata`

> 另有 `AIMessageChunk`(流式分片,可 `+` 累加)、`ChatMessage`(自定义 role)、`FunctionMessage`(已废弃,用 ToolMessage)。

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧱 第一部分:怎么定义

</div>

#### 1.1 四种写法(等价)

In [ ]:
from langchain_core.messages import (
    SystemMessage, HumanMessage, AIMessage, ToolMessage, RemoveMessage,
)

# ① 类(最明确,可带 name / id / 元数据)
m1 = HumanMessage(content="你好", name="bohao", id="h1")

# ② 二元组 ("role", "content")  —— 最短,常用于 invoke / prompt
m2 = ("human", "你好")

# ③ OpenAI 字典格式  —— 兼容外部数据 / JSON
m3 = {"role": "user", "content": "你好"}

# ④ 纯字符串  —— 自动当成 HumanMessage
m4 = "你好"

# 统一规范化成 Message 对象
from langchain_core.messages import convert_to_messages
convert_to_messages([m2, m3, m4])

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**:②③④ 只是**语法糖**,进入 model / graph 前都会被 `convert_to_messages` 转成类实例。role 别名:`user=human` · `assistant=ai` · `system` · `tool`。

</div>

#### 1.2 `content` 的两种形态

- **`str`**:纯文本
- **`list[dict]`**:content blocks(多模态 / 带推理块 / Anthropic 风格)

`.text` 属性:统一取出纯文本(忽略非文本块)。

In [ ]:
img = HumanMessage(content=[
    {"type": "text", "text": "这张图里有什么?"},
    {"type": "image", "url": "https://example.com/cat.png"},   # 也可 base64 + mime_type
])
print(img.text)   # 仅文本部分

#### 1.3 `AIMessage` —— 带工具调用(手写常用于 few-shot / 测试 / 回放历史)

In [ ]:
ai = AIMessage(
    content="",
    tool_calls=[{"name": "multiply", "args": {"a": 2, "b": 3}, "id": "call_1"}],
)
ai.tool_calls          # [{'name','args','id','type'}]  已解析好的 dict
# 模型真实返回时还会有:
#   ai.usage_metadata     -> {'input_tokens','output_tokens','total_tokens'}
#   ai.response_metadata  -> 模型名 / finish_reason 等 provider 信息
#   ai.invalid_tool_calls -> JSON 解析失败的调用

#### 1.4 `ToolMessage` —— 工具结果,必须回指 `tool_call_id`

In [ ]:
tm = ToolMessage(
    content="6",
    tool_call_id="call_1",      # ← 必须等于 AIMessage.tool_calls[i]["id"]
    name="multiply",
    # artifact=...              # 给下游用、不发给模型的原始数据
    # status="error"            # 默认 "success"
)

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**不对齐的后果**:`AIMessage` 里有 `tool_calls`,后面却缺对应 `ToolMessage`(或 `tool_call_id` 对不上)→ provider 直接报 400。**每个 tool_call 必须恰好有一条回应**。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🚚 第二部分:怎么传参

</div>

#### 2.1 喂给 model:`invoke(list)`;返回永远是 `AIMessage`

In [ ]:
from langchain.chat_models import init_chat_model
llm = init_chat_model("openai:gpt-4o")      # 任意 chat model

msgs = [
    SystemMessage("你是简洁的助手"),
    HumanMessage("1+1=?"),
]
reply = llm.invoke(msgs)                     # -> AIMessage
reply = llm.invoke("1+1=?")                  # str  -> 自动包成 HumanMessage
reply = llm.invoke([("system", "简洁"), ("human", "1+1=?")])   # 元组列表

#### 2.2 工具调用闭环:AI → Tool → AI

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply a and b."""
    return a * b

llm = init_chat_model("openai:gpt-4o")

llm_t = llm.bind_tools([multiply])           # multiply 为 @tool 函数

ai = llm_t.invoke([HumanMessage("2*3=?")])   # AIMessage(tool_calls=[...])
history = [HumanMessage("2*3=?"), ai]

for tc in ai.tool_calls:
    history.append(multiply.invoke(tc))      # 传入 tool_call dict → 直接返回 ToolMessage ✨

final = llm_t.invoke(history)                # 带上完整历史再问一次

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**:`tool.invoke(tool_call_dict)` 会**自动生成 `ToolMessage` 并填好 `tool_call_id`**;只传 `args` 则只返回裸值。

</div>

#### 2.3 放进 Prompt:`ChatPromptTemplate` + `MessagesPlaceholder`

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是 {role}"),
    MessagesPlaceholder("history"),          # 把一整个 list[Message] 插进来
    ("human", "{question}"),
])
prompt.invoke({"role": "助教", "history": history, "question": "总结一下"})

#### 2.4 放进 LangGraph State:`add_messages` reducer

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import MessagesState
from langgraph.graph.message import add_messages

# 写法 A:内置(最省事,自带 messages 字段)
class State(MessagesState):
    summary: str                              # 可扩展自己的字段

# 写法 B:手写(等价)
class State2(TypedDict):
    messages: Annotated[list, add_messages]

# 节点只需返回「增量」,reducer 负责合并
def node(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**`add_messages` 的三条规则**
- **追加**:新消息 id 不存在 → append
- **覆盖**:id 相同 → 替换原消息
- **删除**:返回 `RemoveMessage(id=...)` → 从 state 删掉
- 输入里的 tuple / dict / str 也会被自动转成 Message,并**自动补 id**

</div>

In [ ]:
# 删除旧消息(控制 context 长度)
def trim(state: State):
    return {"messages": [RemoveMessage(id=m.id) for m in state["messages"][:-2]]}

#### 2.5 消息管理工具箱(都在 `langchain_core.messages`)

| 函数 | 作用 |
|---|---|
| `trim_messages` | 按 token / 条数裁剪(`strategy="last"`,`max_tokens`,`token_counter`) |
| `filter_messages` | 按 `include_types` / `exclude_names` / `include_ids` 过滤 |
| `merge_message_runs` | 合并**连续同类型**消息(某些 provider 不允许连续同角色) |
| `convert_to_messages` | 任意格式 → Message 对象 |
| `messages_to_dict` / `messages_from_dict` | 序列化 / 反序列化(存库) |
| `get_buffer_string` | 转成 `Human: ... / AI: ...` 文本 |

In [ ]:
from langchain_core.messages import trim_messages, filter_messages, merge_message_runs

trim_messages(history, strategy="last", max_tokens=2, token_counter=len,   # len=按条数
              start_on="human", include_system=True)
filter_messages(history, include_types=[HumanMessage, AIMessage])
merge_message_runs([HumanMessage("a"), HumanMessage("b")])   # -> 一条 "a\nb" 

#### 2.6 流式:`AIMessageChunk` 可直接相加

In [ ]:
full = None
for chunk in llm.stream("讲个笑话"):          # 每个 chunk 是 AIMessageChunk
    full = chunk if full is None else full + chunk
type(full)   # AIMessageChunk;累加后 tool_calls / usage 也会合并

### 📋 速查:谁能放进哪里

| 场景 | 接受的形态 |
|---|---|
| `llm.invoke(...)` | `str` · `list[Message \| tuple \| dict]` |
| `ChatPromptTemplate.from_messages` | tuple · Message · `MessagesPlaceholder` |
| LangGraph `messages` state | 任意形态(自动转换) · `RemoveMessage` |
| 节点返回值 | `{"messages": [增量]}` —— **别返回完整历史** |
| 模型输出 | 恒为 `AIMessage`(流式为 `AIMessageChunk`) |

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**:Message = `role(类型) + content + 元数据`;**四种写法等价**,进图前自动规范化;**工具链靠 `tool_call_id` 配对**,**state 靠 `id` + `add_messages` 增删改**。

</div>